# Phase 6.5 shard 16 (forest65)

Runs **108 cells** of the frozen Phase 6.5 manifest (`G3-PHASE65-v1`), covering: `causal_drf`, `causal_drf_log`, `causal_drf_retn`, `drf`, `drf_log`.

This shard runs the R forest baselines, including the two adversarial controls (log geometry and bandwidth retune). The setup cell installs R, the pinned `drf` 1.3.1, and the authors' causal-clean package at the frozen commit; fifteen to twenty-five minutes.

Estimated single-threaded compute on the reference machine is about **87 minutes**. Colab cores are slower, so allow two to three times that, plus any install time above. This fits comfortably inside a nine hour session.

**Run every cell in order.** The last cell downloads a `.zip`; collect every shard's zip into `results/phase65/colab_shards/` (logs into `results/manifests/`) and run `python research/run_phase65.py merge`.


In [ ]:
# Thread pinning MUST happen before NumPy or SciPy are imported.
# OpenMP sizes its pool at initialisation, so setting these
# afterwards is silently ineffective.
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS', 'R_NUM_THREADS'):
    os.environ[_v] = '1'
print('threads pinned to 1')

## 1. Clone the repository at the pinned commit

Remote `https://github.com/hugogobato/wasserstein-causal-forests.git`, commit `bfe99cb3f305`. After checkout the notebook asserts the frozen manifest checksum, so a clone of anything but the generating commit fails here rather than mid-run.

In [ ]:
import subprocess, pathlib, os, sys, json, hashlib

REPO = 'https://github.com/hugogobato/wasserstein-causal-forests.git'
COMMIT = 'bfe99cb3f305d5f9372dfb723888ed128b8f6ed9'
EXPECTED_CHECKSUM = '4e28d308ca99cde4c81379524fc4492a15b38f029b449899b0a307b6c0ace110'

workdir = pathlib.Path('/content/wcf')
if not workdir.exists():
    subprocess.run(['git', 'init', '-q', str(workdir)], check=True)
    subprocess.run(
        ['git', '-C', str(workdir), 'remote', 'add', 'origin', REPO],
        check=True,
    )
# A shallow fetch of the exact commit: nothing else is downloaded.
    subprocess.run(
        ['git', '-C', str(workdir), 'fetch', '-q', '--depth', '1',
         'origin', COMMIT], check=True,
    )
    subprocess.run(
        ['git', '-C', str(workdir), 'checkout', '-q', 'FETCH_HEAD'],
        check=True,
    )
os.chdir(workdir)
sys.path.insert(0, str(workdir / 'src'))
os.environ['WCF_CAUSAL_DRF_R_LIB'] = '/content/wcf/results/Rlib/causal_drf'

manifest = json.load(open(
    'results/manifests/phase65_manifest.json', encoding='utf-8'
))
checksum = hashlib.sha256(
    json.dumps(manifest['cells'], sort_keys=True).encode('utf-8')
).hexdigest()
assert checksum == EXPECTED_CHECKSUM, (
    'the cloned manifest does not match the frozen grid: '
    f'{checksum} != {EXPECTED_CHECKSUM}'
)
print('repo ready at commit ' + COMMIT[:12] + '; '
      + str(manifest['n_cells']) + ' frozen cells verified')

## 2. Dependencies

In [ ]:
# This group runs the R forest baselines, including Causal-DRF
# through the authors' causal-clean package at the frozen commit.
# The causal-clean repository is a monorepo whose R package sits in
# r-package/drf, so the installer fetches the exact-commit tarball
# from codeload (no GitHub API, hence no shared-IP rate limit) and
# runs R CMD INSTALL on that subdirectory. Expect fifteen to twenty-
# five minutes for this cell.
%%bash
set -e
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev curl > /dev/null 2>&1
Rscript -e 'options(Ncpus=2); install.packages(c("Rcpp","RcppEigen","jsonlite","remotes","transport","fastDummies","kernlab"), repos="https://cloud.r-project.org", quiet=TRUE)'
# CRAN drf 1.3.1 drives the paper-DRF and W-DRF-T drivers; the
# causal-clean library below shadows it only for Causal-DRF cells.
Rscript -e 'options(Ncpus=2); if (!requireNamespace("drf", quietly=TRUE)) install.packages("drf", repos="https://cloud.r-project.org", quiet=TRUE); cat("CRAN drf", as.character(packageVersion("drf")), "ready\n")'
mkdir -p results/Rlib/causal_drf
CAUSAL_SHA="0a1a508444176b5b1553f13e832be93a374b0af2"
if [ ! -d results/Rlib/causal_drf/drf ]; then
  TARBALL="/tmp/causal_clean_${CAUSAL_SHA:0:12}.tar.gz"
  curl -sL "https://codeload.github.com/herbps10/drf/tar.gz/${CAUSAL_SHA}" -o "$TARBALL"
  EXTRACT="/tmp/causal_clean_src"
  rm -rf "$EXTRACT"; mkdir -p "$EXTRACT"
  tar -xzf "$TARBALL" -C "$EXTRACT"
  PKG_DIR=$(find "$EXTRACT" -maxdepth 3 -type d -path "*r-package/drf" | head -1)
  echo "installing causal-clean drf from $PKG_DIR"
  R CMD INSTALL --library=results/Rlib/causal_drf "$PKG_DIR" \
    || Rscript -e 'options(Ncpus=2); .libPaths(c("results/Rlib/causal_drf",.libPaths())); remotes::install_github("herbps10/drf", ref="0a1a508444176b5b1553f13e832be93a374b0af2", subdir="r-package/drf", lib="results/Rlib/causal_drf", upgrade="never", quiet=TRUE)'
fi
Rscript -e '.libPaths(c("results/Rlib/causal_drf",.libPaths())); stopifnot(requireNamespace("drf", quietly=TRUE)); cat("causal-clean drf", as.character(packageVersion("drf")), "ready\n")'
echo 'setup complete'


## 3. This shard's cells

In [ ]:
import json, collections
SHARD_INDEX = 16
CELLS = json.loads('''[{"grid": "c_scaling", "dgp": "IC0", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "26bafe2a4bca7323", "test_seed": 900004}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "b978bb31732d8d2c", "test_seed": 900004}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "6ca84e93c9e1837a", "test_seed": 900004}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "ef59e665e07b52b6", "test_seed": 900004}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "4e74fb0418d02446", "test_seed": 900004}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "fb481e1989403e5b", "test_seed": 900004}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "22899e8b7cc75150", "test_seed": 900004}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "934a8c1a7415d2bb", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 4, "cell_key": "4ee1f28abd14afb1", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 9, "cell_key": "8fb6a26fbdba5bbd", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 4, "cell_key": "797c8c994d559874", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 9, "cell_key": "26c43f00ac041314", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 4, "cell_key": "47e22df4449dd9d8", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 9, "cell_key": "1c035796daa23bff", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 4, "cell_key": "174178126a5752d4", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 9, "cell_key": "43e5fe8a3220bdb7", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 4, "cell_key": "695b940cb48c5781", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 9, "cell_key": "58492e51a2004fe1", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 4, "cell_key": "7f2669538c84e2d9", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 9, "cell_key": "bd61530c2f803507", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 4, "cell_key": "b9d66e4692cdd04c", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 9, "cell_key": "4656ff090793e4f6", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 4, "cell_key": "5e7025a76bc0f816", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 9, "cell_key": "ec7a6a4041aaae17", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 4, "cell_key": "60705766a772fead", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 9, "cell_key": "76c26fb8990e2af2", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 4, "cell_key": "0c60636d776b479e", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 9, "cell_key": "d688199672dd0fde", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 4, "cell_key": "99336896309ee901", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 9, "cell_key": "49300ed74958adfa", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 4, "cell_key": "d0df73c399508b51", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 9, "cell_key": "ccf5183b94deade7", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 4, "cell_key": "4e697f221d6fefcb", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 9, "cell_key": "d44e6a062555510c", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 4, "cell_key": "cc9f7cfaa99ee9ca", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 9, "cell_key": "ab3647cf6fcc61ef", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 4, "cell_key": "3e8d361b3aa21730", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 9, "cell_key": "b3fff2b65be5ef73", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 4, "cell_key": "64020841e8ca8835", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 9, "cell_key": "a20763b89bebe21f", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "96aa071dae37bfa9", "test_seed": 900004}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "936fb741d2263b8b", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "2217e9f4bd469dda", "test_seed": 900004}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "e99c5b92cba900e8", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "4ef8dd7f3a69f838", "test_seed": 900004}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "b73cd3d43ca218a6", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "411df39c55d564f0", "test_seed": 900004}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "94a209524498d4f6", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "7c705309adea30b8", "test_seed": 900004}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "3aeb7b71118bc043", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "da493b7b70e5c66f", "test_seed": 900004}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "8faf8fbac2a8e9a9", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "bdbed47c50e3c75a", "test_seed": 900004}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "562e5851962054aa", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "5338ee40fbfd2d1f", "test_seed": 900004}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "5ff15e78a91c6f45", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "2e2a25f4d8efe957", "test_seed": 900004}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "ade082a08701c7b3", "test_seed": 900009}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "9d6ae3e559cecd50", "test_seed": 900004}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "de2c2ec749a30ce7", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "c07eb33c6ed8552d", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "a2f13688aa328b04", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "2ae34db7c531f1e1", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "2101087674236841", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "dee73a587174ddca", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "b6c60144a55452ba", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "e6f6a5f21e96f217", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "d389ae9abb340e46", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "5144f1f5294ba431", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "6b71830668858bda", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "d33bfdc7f88d30c5", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "8fcdeceabddb7d40", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "730f672dfad8a0ba", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "be4fcf7314923db4", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "f6bdd8f0bb928a7a", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "706e3f7ff04e0ac1", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 4, "cell_key": "4994692757d47a3f", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 9, "cell_key": "81cf7684933dbe89", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 4, "cell_key": "3c91236d0f89774c", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 9, "cell_key": "061580d2c9523f8f", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 4, "cell_key": "a1dce948c0bd8beb", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 9, "cell_key": "136c623e067a0958", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 4, "cell_key": "ac76fbfc927aea0b", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 9, "cell_key": "f5f0a87f2f20d298", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 4, "cell_key": "217841b0704a3580", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 9, "cell_key": "92427b0658dca6dd", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 4, "cell_key": "396a7ec8d9bee7ef", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 9, "cell_key": "930712e70560f8c1", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 4, "cell_key": "791117bd60a95a18", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 9, "cell_key": "eb4e3d6f4f5086fc", "test_seed": 900009}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 4, "cell_key": "ac09e4e5dd1afe04", "test_seed": 900004}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 9, "cell_key": "02cf7819dcfa085e", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "dc291c1a06e31fb7", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "7ca9e4271539b1d8", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "0e4468e44eb124bb", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "be27de44926fcaeb", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "23ac678893d6b892", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "b67064af5a2376e4", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "12be0cbcbb36dcf0", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "c71eb513c6f72b3a", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "1d8570573cde2a46", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "97eb3bdd96df2e35", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "8eb39b5f452677ff", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "82c4d5e4d2f7fe4a", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 4, "cell_key": "8a14ee2dd30cf29f", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 9, "cell_key": "cc8edd8759ebb400", "test_seed": 900009}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 4, "cell_key": "fffec0ca963f4350", "test_seed": 900004}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 9, "cell_key": "eeb5f6c8c8d267fe", "test_seed": 900009}]''')
print(f'{len(CELLS)} cells in this shard')
for key, count in sorted(collections.Counter(
        (c['grid'], c['dgp'], c['method'])
        for c in CELLS).items()):
    print(f'  {key[0]:12s} {key[1]:8s} {key[2]:18s} {count}')

## 4. Bandwidth-selection pilot (preregistered)

This shard contains `causal_drf_retn` cells, so it first runs the selection pilot on seeds 100 and 101, outside every decisive range, and freezes the multipliers document. The rule picks the candidate with the best held-out energy score; oracle truth is never read.

In [ ]:
from pathlib import Path
import json, numpy as np
from wasserstein_causal_forests.g3.dgps import build_dgp
from wasserstein_causal_forests.g3.phase65_methods import (
    BANDWIDTH_CANDIDATES, SELECTION_SEEDS, select_bandwidth_multiplier,
)

keys = sorted({(c['dgp'], c['n_train']) for c in CELLS
               if c['method'] == 'causal_drf_retn'})
multipliers = {}
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)
for dgp_name, n_train in keys:
    dgp = build_dgp(dgp_name, 25)
    best, means = select_bandwidth_multiplier(
        dgp, n_train, seeds=SELECTION_SEEDS,
        candidates=BANDWIDTH_CANDIDATES, cache_directory=cache,
    )
    multipliers[f'{dgp_name}|{n_train}'] = best
    scores = {str(k): round(v, 5) for k, v in means.items()}
    print(f'{dgp_name} n={n_train}: multiplier {best}  scores {scores}',
          flush=True)

document = {
    'rule': 'held-out energy score, pilot seeds 100 and 101, '
            'candidates ' + repr(BANDWIDTH_CANDIDATES),
    'multipliers': multipliers,
}
path = Path('/content/wcf/results/manifests/'
            'phase65_bandwidth_selection.json')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(document, indent=2))
print('froze', path)

## 5. Run

In [ ]:
import time
from pathlib import Path
from wasserstein_causal_forests.g3.manifest import Cell
from wasserstein_causal_forests.g3.runner import run_shard

cells = [Cell(**{k: v for k, v in item.items()
                 if k not in ('cell_key', 'test_seed')})
         for item in CELLS]

out = Path('/content/wcf/results/phase65/colab_shards')
out.mkdir(parents=True, exist_ok=True)
log = Path(f'/content/wcf/results/manifests/phase65_execution_log_{SHARD_INDEX:03d}.jsonl')
log.parent.mkdir(parents=True, exist_ok=True)
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)

started = time.time()
summary = run_shard(
    cells,
    out / f'shard_{SHARD_INDEX:03d}.parquet',
    cache_directory=cache,
    log_path=log,
    manifest_contract_id='G3-PHASE65-v1',
)
print(json.dumps(summary, indent=2))
print(f'elapsed {(time.time() - started) / 60:.1f} min')

## Check

Every cell must appear exactly once, as a success or as a failure. Failures are kept and reported at merge time; a seed is never silently replaced.

In [ ]:
import collections
records = [json.loads(line) for line in
           open(log, encoding='utf-8') if line.strip()]
status = collections.Counter(r['status'] for r in records)
print('cells logged:', len(records), '| expected:', len(CELLS))
print('status:', dict(status))
assert len(records) == len(CELLS), 'shard did not finish every cell'
for record in records:
    if record['status'] != 'ok':
        print('  FAILED', record['dgp'], record['method'],
              record['seed'])
slowest = sorted(records, key=lambda r: -r['wall_seconds'])[:5]
print('slowest cells:', [(r['method'], round(r['wall_seconds'], 1))
                         for r in slowest])

## Download the results

In [ ]:
import shutil
bundle = '/content/p65_shard_16_forest65'
staging = Path('/content/bundle')
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)
shutil.copy(out / f'shard_{SHARD_INDEX:03d}.parquet', staging)
if log.exists():
    shutil.copy(log, staging)
output_file = shutil.make_archive(bundle, 'zip', staging)
print('bundle:', output_file,
      f'({os.path.getsize(output_file) / 1e6:.2f} MB)')

try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)